# Chunks → transcription → dataset
One notebook from the diarization to the finished dataset. It needs only the RTTMs from `01_diarization` and the audio. The old transcripts and word alignments are **not used**: every chunk is transcribed again on its own, so its text is what is in its wav.

**Paths:** everything is read and written in the folder that `RUN` in `pipeline_paths.py` sets: the project root for the main dataset, or `runs/<RUN>/` for a separate test run. The audio comes from `data/raw_audios/` (`AUDIOS` there picks which).

Run the cells from top to bottom. Every step skips what it has already done, so after a stop you can run it again.

**1. Turns** (`outputs/chunks/<name>/turns.json`, `short/`)
- The diarization segments of one speaker that follow each other become one **turn**, with the pauses between them. A pause longer than `MAX_PAUSE_SEC` ends the turn and is dropped, except `EDGE_SEC` on each side.
- Two turns are cut apart at the quietest moment of the pause between them (measured on the audio, 10 ms frames), so words are not cut in half.
- Turns shorter than `SHORT_SEC` are saved to `short/`, to be transcribed in step 2.

**2. Transcribe the short turns** (`short_text.json`): this gives their word count.

**3. Omit, merge, split** (`wavs/`, `chunks.json`)
- **Omitted:** a short turn with fewer than `MIN_WORDS` words, such as "ha" or "mm", and diarization blips under 0.1 s.
- **Merged:** when an omitted turn interrupts a speaker (A: "...", B: "ha", A: "..."), its audio is cut out and the two parts of A become one chunk: they belong to one sentence. The two parts are joined with a short crossfade (`JOIN_FADE_SEC`). A chunk like this is shorter than `end - start`: its `parts` say which stretches of the audio it holds.
- A short turn with enough words stays as a chunk of its own, and the speech around it is not merged.
- **Split:** a chunk longer than `MAX_CHUNK_SEC` is split at its longest pauses. A cut-out interruption counts as a pause too.

**4. Transcribe the chunks** (`text.json`): with the Whisper API, the same server as `src/transcribe.ipynb`. A short turn that stays a chunk of its own is the same audio that step 2 transcribed, so that text is reused (in the main dataset 179 chunks, and 178 of them came back the same both times).

**5. Dataset** (`dataset/chunks/`, `dataset/metadata.json`)
- **Omitted:** an empty transcript; a word or phrase said more than `MAX_REPEATS` times in a row (a Whisper loop: `shu shu shu shu`, `O'o'o'o'o'o'`).
- **Apostrophe fix:** `bo 'ladi` becomes `bo'ladi`.
- Wavs are resampled to 48 kHz. `metadata.json` is `{sample_rate, podcasts, chunks: [{file, speaker, text, duration, start, end}]}`.

In [ ]:
import json
import math
import os
import re
import time
import traceback
import uuid
from bisect import bisect_left, bisect_right
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib import error, parse, request

import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm.auto import tqdm

import importlib
import pipeline_paths as P
importlib.reload(P)   # a change of RUN in pipeline_paths.py counts without restarting the kernel

root = P.ROOT
out_root = P.OUTPUTS
audio_dir = P.AUDIO_DIR
chunks_root = out_root / "chunks"   # steps 1-4: <name>/turns.json, short/, short_text.json, wavs/, chunks.json, text.json
dataset_dir = P.DATASET            # step 5
wav_dir = dataset_dir / "chunks"
print(P.describe())
wavs = P.audio_files()
for i, p in enumerate(wavs):
    print(i, p.name)

In [ ]:
# 1. turns
ONLY = None           # None = every audio; or a list of indices from the list above, e.g. [3, 5]
FORCE = False         # True = cut the turns of these audios again (and redo steps 2-4 for them)
MAX_PAUSE_SEC = 3.0   # a pause longer than this ends a turn, even if the same speaker goes on; it is dropped...
EDGE_SEC = 0.5        # ...except this much of it on each side
CUT_SLACK_SEC = 0.1   # a cut between two turns may go this far into a turn if it is quieter there (diarization is coarse)
SHORT_SEC = 1.0       # a turn shorter than this is short: transcribed in step 2 (changing it needs FORCE = True)
FADE_SEC = 0.01       # fade in / out at the wav edges, against clicks

# 2 and 4. transcription
API = "http://192.168.1.71:8199"
LANGUAGE = "uz"       # every podcast was detected as uz; "auto" guesses badly on 1-second clips
WORKERS = 8           # chunks in flight; the server decodes 4 at once, the rest wait in its queue
POLL_SECONDS = 1.0    # how often to ask the server whether a chunk is done
JOB_TIMEOUT = 3600    # seconds per chunk, waiting in the server's queue included
SAVE_EVERY = 50       # save the transcripts after this many chunks

# 3. omit, merge, split
MIN_WORDS = 3         # a short turn with fewer words than this is omitted (2 = keep the two-word ones)
FORCE_MERGE = False   # True = redo step 3 (e.g. after changing MIN_WORDS); step 4 then transcribes again
MAX_CHUNK_SEC = 29.5  # longest chunk: Whisper hears 30 s at a time, a longer chunk would spill into a second window
MIN_SPLIT_SEC = 2.0   # splitting a long chunk never leaves a piece shorter than this
JOIN_FADE_SEC = 0.02  # crossfade where an omitted interruption was cut out

# 5. dataset
MAX_REPEATS = 3       # a word or phrase (up to 4 words) said more often than this in a row = Whisper loop, omitted
OUT_SR = 48000
# sharp filter: torchaudio's default one takes ~3-4 dB off the top of the band, this one keeps it
RESAMPLE = dict(lowpass_filter_width=64, rolloff=0.99, resampling_method="sinc_interp_kaiser", beta=14.769656459379492)

assert MAX_PAUSE_SEC > 2 * EDGE_SEC and MAX_CHUNK_SEC > 2 * MIN_SPLIT_SEC

## 1. Turns

In [ ]:
RES = 100                    # grid: frames per second (10 ms)
SILENCE, OVERLAP = -1, -2    # frame labels; 0, 1, 2, ... = index of the one speaker talking
MIN_WAV_SEC = 0.1            # a turn shorter than this is a diarization blip: omitted without transcribing it


def read_rttm(path):
    segments = []
    for line in path.read_text().splitlines():
        p = line.split()  # counted from the end, so an id with spaces still works
        s = float(p[-7])
        segments.append((s, s + float(p[-6]), p[-3]))
    return sorted(segments)


def frame_labels(segments, n):
    """speakers, labels[frame]: index of the one speaker talking, SILENCE or OVERLAP."""
    speakers = sorted({spk for *_, spk in segments})
    active = np.zeros((len(speakers), n), dtype=bool)
    for s, e, spk in segments:
        active[speakers.index(spk), max(0, round(s * RES)):min(n, round(e * RES))] = True
    count = active.sum(axis=0)
    labels = np.full(n, SILENCE, dtype=np.int16)
    labels[count > 1] = OVERLAP
    if speakers:
        one = count == 1
        labels[one] = active[:, one].argmax(axis=0)
    return speakers, labels


def runs(labels):
    """(label, start, end) of every stretch of equal labels, in frames."""
    change = np.flatnonzero(np.diff(labels)) + 1
    starts, ends = np.r_[0, change], np.r_[change, len(labels)]
    return [(int(labels[a]), int(a), int(b)) for a, b in zip(starts, ends)]


def loudness(path, n):
    """Power of every 10 ms frame of the audio (mono), averaged over 50 ms. Low = quiet."""
    sr = sf.info(str(path)).samplerate
    power, count = np.zeros(n), np.zeros(n)
    pos = 0
    for block in sf.blocks(str(path), blocksize=sr * 60, dtype="float32", always_2d=True):
        x = block.mean(axis=1).astype(np.float64)
        frame = np.minimum(np.arange(pos, pos + len(x)) * RES // sr, n - 1)
        f0, f1 = frame[0], frame[-1] + 1
        power[f0:f1] += np.bincount(frame - f0, weights=x * x)
        count[f0:f1] += np.bincount(frame - f0)
        pos += len(x)
    power /= np.maximum(count, 1)
    return np.convolve(power, np.ones(5) / 5, mode="same")


def analyse(audio_path):
    """speakers, frame labels, loudness and length (s) of one audio."""
    name = audio_path.stem
    rttm_path = out_root / "diarization" / name / "nemotron" / f"{name}.rttm"
    if not rttm_path.exists():
        raise FileNotFoundError(f"no RTTM: run 01_diarization on {audio_path.name} first")
    total_sec = sf.info(str(audio_path)).duration
    n = math.ceil(total_sec * RES)
    speakers, labels = frame_labels(read_rttm(rttm_path), n)
    return speakers, labels, loudness(audio_path, n), total_sec


def quietest(power, lo, hi):
    """Time (s) of the quietest moment between lo and hi. Of the frames within 1 dB of the quietest, the one
    nearest the middle: in a silent pause the cut lands in its middle, not right after a word."""
    a, b = max(0, math.ceil(lo * RES)), min(len(power), math.floor(hi * RES) + 1)
    if b <= a:
        return (lo + hi) / 2
    p = power[a:b]
    near = np.flatnonzero(p <= p.min() * 10 ** 0.1 + 1e-12)
    best = near[np.argmin(np.abs(near - ((lo + hi) / 2 * RES - a)))]
    return float(min(max((a + best + 0.5) / RES, lo), hi))


def speaker_turns(labels):
    """[speaker, first frame, end frame] of every turn: the segments of one speaker that follow each other, with the
    pauses (and overlaps) between them. A pause longer than MAX_PAUSE_SEC ends the turn."""
    long_pause = MAX_PAUSE_SEC * RES
    turns = []
    for lab, a, b in runs(labels):
        if lab < 0:
            continue
        prev = turns[-1] if turns else None
        if prev and prev[0] == lab and not (a - prev[2] > long_pause and (labels[prev[2]:a] == SILENCE).all()):
            prev[2] = b
        else:
            turns.append([lab, a, b])
    return turns


def turn_bounds(turns, labels, power, total_sec):
    """(start, end) in seconds of every turn.
    Between two turns there is one cut, at the quietest moment of the pause (or overlap) between them; it may go
    CUT_SLACK_SEC into a turn, never past the turn's middle. Before the first turn, after the last one and in a
    pause longer than MAX_PAUSE_SEC each turn keeps EDGE_SEC of the pause; the rest of it is dropped."""
    long_pause = MAX_PAUSE_SEC * RES
    first, last = turns[0][1], turns[-1][2]
    starts, ends = [first / RES - EDGE_SEC if first > long_pause else 0.0], []
    for (_, a1, b1), (_, a2, b2) in zip(turns, turns[1:]):
        if a2 - b1 > long_pause and (labels[b1:a2] == SILENCE).all():
            ends.append(b1 / RES + EDGE_SEC)
            starts.append(a2 / RES - EDGE_SEC)
        else:
            t = quietest(power, max(b1 / RES - CUT_SLACK_SEC, (a1 + b1) / 2 / RES),
                         min(a2 / RES + CUT_SLACK_SEC, (a2 + b2) / 2 / RES))
            ends.append(t)
            starts.append(t)
    ends.append(last / RES + EDGE_SEC if len(labels) - last > long_pause else total_sec)
    return list(zip(starts, ends))


def read_parts(f, parts):
    """The audio of parts [(start, end), ...] of an open SoundFile, mono, joined with a JOIN_FADE_SEC crossfade,
    with a FADE_SEC fade at both ends."""
    sr = f.samplerate
    x = None
    for s, e in parts:
        a, b = round(s * sr), min(round(e * sr), f.frames)
        f.seek(a)
        y = f.read(b - a, dtype="float32", always_2d=True).mean(axis=1)
        if x is None:
            x = y
            continue
        n = min(round(JOIN_FADE_SEC * sr), len(x) // 2, len(y) // 2)
        ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
        x = np.concatenate([x[:len(x) - n], x[len(x) - n:] * ramp[::-1] + y[:n] * ramp, y[n:]])
    m = min(round(FADE_SEC * sr), len(x) // 2)
    if m:
        ramp = np.linspace(0.0, 1.0, m, dtype=np.float32)
        x[:m] *= ramp
        x[-m:] *= ramp[::-1]
    return x


def cut_turns(audio_path):
    """Step 1 for one audio: turns.json, and the short turns' wavs in short/."""
    out_dir = chunks_root / audio_path.stem
    path = out_dir / "turns.json"
    if path.exists() and not FORCE:
        print("  done before, skipping")
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in ("turns.json", "short_text.json", "chunks.json", "text.json", "manifest.jsonl", "summary.json"):
        (out_dir / old).unlink(missing_ok=True)  # new turns: everything made from the old ones is out of date
    for sub in ("short", "wavs"):
        for old in (out_dir / sub).glob("*.wav"):
            old.unlink()

    speakers, labels, power, total_sec = analyse(audio_path)
    turns = speaker_turns(labels)
    bounds = turn_bounds(turns, labels, power, total_sec) if turns else []
    (out_dir / "short").mkdir(exist_ok=True)
    records = []
    with sf.SoundFile(audio_path) as f:
        for k, ((lab, _, _), (s, e)) in enumerate(zip(turns, bounds)):
            s, e = round(s, 3), round(e, 3)
            file = None
            if MIN_WAV_SEC <= e - s < SHORT_SEC:
                file = f"short/{k:04d}_{speakers[lab]}.wav"
                sf.write(out_dir / file, read_parts(f, [(s, e)]), f.samplerate, subtype="PCM_16")
            records.append({"speaker": speakers[lab], "start": s, "end": e, "file": file})

    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps({"audio": audio_path.name, "duration": round(total_sec, 3), "turns": records},
                              ensure_ascii=False, indent=1), encoding="utf-8")
    os.replace(tmp, path)  # written last = done marker
    print(f"  {len(records)} turns: {sum(r['file'] is not None for r in records)} short (go to step 2), "
          f"{sum(r['end'] - r['start'] < MIN_WAV_SEC for r in records)} blips under {MIN_WAV_SEC} s (omitted)")

In [ ]:
selected = wavs if ONLY is None else [wavs[i] for i in ONLY]
failed = {}
for i, audio_path in enumerate(selected, 1):
    print(f"[{i}/{len(selected)}] {audio_path.name}")
    try:
        cut_turns(audio_path)
    except Exception:
        failed[audio_path.name] = traceback.format_exc()
        print("  FAILED (continuing)")

print(f"\ndone: {len(selected) - len(failed)}, failed: {len(failed)}")
for name, tb in failed.items():
    print("\n" + name + "\n" + tb)

## 2. Transcribe the short turns
The code here is used for step 4 too. A chunk that fails (server down, timeout) is listed at the end: run the cell again to retry it. Finished chunks are not sent again.

In [ ]:
DONE = {"done", "completed", "complete", "succeeded", "success", "finished"}
FAILED = {"failed", "error", "errored", "cancelled", "canceled"}
APOSTROPHES = "'‘’ʻʼ`"


def http_json(url, *, data=None, headers=None, timeout=60):
    req = request.Request(url, data=data, headers=headers or {})
    try:
        with request.urlopen(req, timeout=timeout) as resp:
            return resp.status, json.loads(resp.read() or b"{}")
    except error.HTTPError as e:
        body = e.read().decode("utf-8", "replace")
        try:
            return e.code, json.loads(body)
        except json.JSONDecodeError:
            return e.code, {"detail": body}


def retry(fn, attempts, what):
    delay = 2.0
    for i in range(1, attempts + 1):
        try:
            return fn()
        except (error.URLError, TimeoutError, ConnectionError, RuntimeError) as e:
            if i == attempts:
                raise RuntimeError(f"{what} failed after {attempts} attempts: {e}") from e
            time.sleep(delay)
            delay = min(delay * 2, 30)


def upload_body(fields, path):
    """multipart/form-data with the fields and the wav (a chunk is small, so it is read into memory)."""
    boundary = uuid.uuid4().hex
    body = b"".join(f'--{boundary}\r\nContent-Disposition: form-data; name="{k}"\r\n\r\n{v}\r\n'.encode()
                    for k, v in fields.items())
    body += (f'--{boundary}\r\nContent-Disposition: form-data; name="file"; filename="audio.wav"\r\n'
             f"Content-Type: audio/wav\r\n\r\n").encode() + path.read_bytes() + f"\r\n--{boundary}--\r\n".encode()
    return body, f"multipart/form-data; boundary={boundary}"


def submit(path):
    def go():
        body, ctype = upload_body({"language": LANGUAGE, "word_probs": "false", "min_probability": "0.0"}, path)
        status, js = http_json(f"{API}/jobs", data=body, headers={"Content-Type": ctype}, timeout=300)
        if status >= 500:
            raise RuntimeError(f"submit HTTP {status}: {js}")  # retried
        if status not in (200, 201, 202):
            raise ValueError(f"submit HTTP {status}: {js}")    # not retried (e.g. 422)
        job_id = js.get("job_id") or js.get("id")
        if not job_id:
            raise ValueError(f"no job id in submit response: {js}")
        return str(job_id)

    return retry(go, 5, "submit")


def wait(job_id):
    deadline = time.monotonic() + JOB_TIMEOUT
    while time.monotonic() < deadline:
        status, js = retry(lambda: http_json(f"{API}/jobs/{parse.quote(job_id)}", timeout=30), 5, "poll")
        if status == 404:
            raise ValueError(f"job {job_id} not found on server (restarted or expired?)")
        if status < 500:
            state = str(js.get("status", "")).lower()
            if state in FAILED:
                raise ValueError(f"job {job_id} failed: {js.get('error') or js}")
            if state in DONE or (not state and ("result" in js or "text" in js or "segments" in js)):
                return js
        time.sleep(POLL_SECONDS)
    raise TimeoutError(f"job {job_id} not finished after {JOB_TIMEOUT}s")


def extract_text(js):
    res = js.get("result", js)
    if isinstance(res, dict):
        if isinstance(res.get("text"), str):
            return res["text"].strip()
        segs = res.get("segments")
        if isinstance(segs, list):
            return " ".join(s.get("text", "").strip() for s in segs if isinstance(s, dict)).strip()
    return ""


def transcribe(path):
    return " ".join(extract_text(wait(submit(path))).split())


def transcribe_all(listing, text_name):
    """Sends every wav of listing ({podcast folder: [wav files in it]}) that has no transcript in
    <folder>/<text_name> yet. Saved every SAVE_EVERY chunks and at the end, also after an error or Interrupt."""
    status, health = http_json(f"{API}/health", timeout=10)
    if status != 200 or health.get("status") != "ready":
        raise RuntimeError(f"API not ready: {health}")
    texts, todo = {}, []
    for d, files in listing.items():
        path = d / text_name
        texts[d] = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
        todo += [(d, f) for f in files if f not in texts[d]]
    print(f"{len(listing)} podcasts: {sum(map(len, listing.values())) - len(todo)} transcribed before, {len(todo)} to send")

    def save(d):
        tmp = d / f"{text_name}.tmp"
        tmp.write_text(json.dumps(dict(sorted(texts[d].items())), ensure_ascii=False, indent=1), encoding="utf-8")
        os.replace(tmp, d / text_name)

    failed, unsaved = {}, set()
    pool = ThreadPoolExecutor(WORKERS)
    try:
        futures = {pool.submit(transcribe, d / f): (d, f) for d, f in todo}
        for k, fut in enumerate(tqdm(as_completed(futures), total=len(futures), unit="chunk"), 1):
            d, f = futures[fut]
            try:
                texts[d][f] = fut.result()
                unsaved.add(d)
            except Exception as e:  # one bad chunk must not stop the run
                failed[f"{d.name}/{f}"] = f"{type(e).__name__}: {e}"
            if k % SAVE_EVERY == 0:
                for x in unsaved:
                    save(x)
                unsaved.clear()
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
        for x in unsaved:
            save(x)
    print(f"done: {len(todo) - len(failed)} transcribed, {len(failed)} failed"
          + (" (run this cell again to retry them)" if failed else ""))
    for name, why in list(failed.items())[:20]:
        print(" ", name, why)
    return failed


def clean(text):
    """One space between words; 'bo 'ladi' -> 'bo'ladi' (the ASR sometimes puts a space before the apostrophe)."""
    return re.sub(rf"(?<=\w) +(?=[{APOSTROPHES}]\w)", "", " ".join(text.split()))


def word_count(text):
    return sum(1 for w in text.split() if re.search(r"\w", w))

In [ ]:
listing = {}
for d in sorted(p for p in chunks_root.iterdir() if (p / "turns.json").exists()):
    turns = json.loads((d / "turns.json").read_text(encoding="utf-8"))["turns"]
    listing[d] = [t["file"] for t in turns if t["file"]]
failed = transcribe_all(listing, "short_text.json")

## 3. Omit, merge, split

In [ ]:
MIN_PART_SEC = 0.05  # a sliver of pause left next to a cut-out interruption is dropped


def omitted_turns(turns, heard):
    """Indices of the omitted turns: blips under MIN_WAV_SEC, and short turns with fewer than MIN_WORDS words."""
    return {i for i, t in enumerate(turns) if t["end"] - t["start"] < MIN_WAV_SEC
            or (t["file"] and word_count(clean(heard[t["file"]])) < MIN_WORDS)}


def merge_groups(turns, omitted):
    """[speaker, [(start, end), ...]] for every group of turns. The omitted turns are dropped; two turns of one
    speaker with nothing but omitted turns between them (A, B: "ha", A) become one group. Not across a long
    pause: that was dropped in step 1, so the times of the turns around it don't touch."""
    groups, last = [], None  # last: index of the previous kept turn
    for i, t in enumerate(turns):
        if i in omitted:
            continue
        if (last is not None and i > last + 1 and turns[last]["speaker"] == t["speaker"]
                and all(turns[k]["end"] == turns[k + 1]["start"] for k in range(last, i))):
            groups[-1][1].append((t["start"], t["end"]))
        else:
            groups.append([t["speaker"], [(t["start"], t["end"])]])
        last = i
    return groups


def pauses(labels, power):
    """(time, length) of every silence in the diarization, in order: where a long chunk can be split.
    time = the quietest moment of that silence."""
    return [(quietest(power, a / RES, b / RES), (b - a) / RES) for lab, a, b in runs(labels) if lab == SILENCE]


def split_points(s, e, cands, times, quiet):
    """Cuts that split s..e into as few pieces as possible, none longer than MAX_CHUNK_SEC or shorter than
    MIN_SPLIT_SEC. Each cut is the longest pause in the window that keeps the number of pieces minimal
    (equal pauses: the one that makes the pieces most even). No pause there: quiet(lo, hi)."""
    if e - s <= MAX_CHUNK_SEC:
        return []
    k = math.ceil((e - s) / MAX_CHUNK_SEC)  # pieces needed
    even = s + (e - s) / k
    lo_t = max(s + MIN_SPLIT_SEC, e - (k - 1) * MAX_CHUNK_SEC)
    hi_t = min(s + MAX_CHUNK_SEC, e - MIN_SPLIT_SEC)
    lo, hi = bisect_left(times, lo_t), bisect_right(times, hi_t)
    t = max(cands[lo:hi], key=lambda c: (round(c[1], 1), -abs(c[0] - even)))[0] if lo < hi else quiet(lo_t, hi_t)
    return [t] + split_points(t, e, cands, times, quiet)


def split_group(parts, cands, times, power):
    """The chunks of one group, each a list of (start, end) parts, none longer than MAX_CHUNK_SEC of audio.
    Works on the group's own clock (its parts end to end); where an interruption was cut out is a pause too."""
    offs = np.cumsum([0.0] + [e - s for s, e in parts])  # where each part starts on the group's clock
    gc = []
    for i, (s, e) in enumerate(parts):
        if i:
            gc.append((offs[i], s - parts[i - 1][1]))  # the cut-out interruption, as long as it was
        gc += [(offs[i] + t - s, length) for t, length in cands[bisect_right(times, s):bisect_left(times, e)]]
    gt = [g for g, _ in gc]

    def quiet(lo, hi):  # a window without any pause lies inside one part
        i = int(np.searchsorted(offs, lo, side="right")) - 1
        s = parts[i][0] - offs[i]
        return quietest(power, lo + s, min(hi, offs[i + 1]) + s) - s

    cuts = [0.0, *split_points(0.0, float(offs[-1]), gc, gt, quiet), float(offs[-1])]
    chunks = []
    for a, b in zip(cuts, cuts[1:]):
        piece = [(round(s + max(a, o0) - o0, 3), round(s + min(b, o1) - o0, 3))
                 for (s, _), o0, o1 in zip(parts, offs, offs[1:]) if min(b, o1) - max(a, o0) >= MIN_PART_SEC]
        chunks.append(piece)
    return chunks


def export(audio_path, chunks, out_dir):
    """The chunks' wavs (mono, original sample rate). Returns (file, duration) per chunk."""
    wav_dir = out_dir / "wavs"
    wav_dir.mkdir(parents=True, exist_ok=True)
    for old in wav_dir.glob("*.wav"):  # an earlier run's wavs
        old.unlink()
    files = []
    with sf.SoundFile(audio_path) as f:
        for k, c in enumerate(chunks):
            name = f"{k:04d}_{c['speaker']}.wav"
            x = read_parts(f, c["parts"])
            sf.write(wav_dir / name, x, f.samplerate, subtype="PCM_16")
            files.append((f"wavs/{name}", round(len(x) / f.samplerate, 3)))
    return files


def check(chunks, files):
    """Raises if parts overlap or are out of order, a chunk is too long, or a wav does not match its parts."""
    flat = [p for c in chunks for p in c["parts"]]
    assert all(s < e for s, e in flat), "empty part"
    assert all(a[1] <= b[0] for a, b in zip(flat, flat[1:])), "parts overlap or are out of order"
    assert len(files) == len(chunks), "a wav is missing"
    assert max((d for _, d in files), default=0) <= MAX_CHUNK_SEC + 0.01, "a wav is longer than MAX_CHUNK_SEC"
    for c, (_, d) in zip(chunks, files):
        expected = sum(e - s for s, e in c["parts"]) - (len(c["parts"]) - 1) * JOIN_FADE_SEC
        assert abs(d - expected) < 0.01, f"wav {d} s, parts {expected} s"


def merge_and_split(d):
    """Step 3 for one podcast folder: wavs/ and chunks.json."""
    path = d / "chunks.json"
    if path.exists() and not FORCE_MERGE:
        print("  done before, skipping")
        return
    record = json.loads((d / "turns.json").read_text(encoding="utf-8"))
    turns = record["turns"]
    heard_path = d / "short_text.json"
    heard = json.loads(heard_path.read_text(encoding="utf-8")) if heard_path.exists() else {}
    missing = [t["file"] for t in turns if t["file"] and t["file"] not in heard]
    if missing:
        raise RuntimeError(f"{len(missing)} short turns are not transcribed yet: run step 2 again")
    for old in ("chunks.json", "text.json"):  # new chunks: their old transcripts don't fit
        (d / old).unlink(missing_ok=True)

    omitted = omitted_turns(turns, heard)
    audio_path = audio_dir / record["audio"]
    _, labels, power, _ = analyse(audio_path)
    cands = pauses(labels, power)
    times = [t for t, _ in cands]
    chunks = [{"speaker": spk, "parts": piece}
              for spk, parts in merge_groups(turns, omitted) for piece in split_group(parts, cands, times, power)]
    files = export(audio_path, chunks, d)
    check(chunks, files)

    out = {
        "audio": record["audio"],
        "duration": record["duration"],
        "chunks": [{"file": file, "speaker": c["speaker"], "start": c["parts"][0][0], "end": c["parts"][-1][1],
                    "duration": duration, "parts": c["parts"]} for c, (file, duration) in zip(chunks, files)],
    }
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding="utf-8")
    os.replace(tmp, path)  # written last = done marker

    lengths = [dur for _, dur in files] or [0]
    print(f"  {len(turns)} turns, {len(omitted)} omitted; {len(chunks)} chunks, "
          f"{sum(len(c['parts']) - 1 for c in chunks)} of them joined across an interruption; "
          f"{sum(lengths) / 60:.1f} min, median {np.median(lengths):.1f} s, max {max(lengths):.1f} s")

In [ ]:
folders = sorted(p for p in chunks_root.iterdir() if (p / "turns.json").exists())
failed = {}
for i, d in enumerate(folders, 1):
    print(f"[{i}/{len(folders)}] {d.name}")
    try:
        merge_and_split(d)
    except Exception:
        failed[d.name] = traceback.format_exc()
        print("  FAILED (continuing)")

print(f"\ndone: {len(folders) - len(failed)}, failed: {len(failed)}")
for name, tb in failed.items():
    print("\n" + name + "\n" + tb)

## 4. Transcribe the chunks

In [ ]:
# A short turn that stayed a chunk of its own is the same audio as its wav in short/, which step 2 transcribed:
# its text is reused instead of sending it again.
listing, reused = {}, 0
for d in sorted(p for p in chunks_root.iterdir() if (p / "chunks.json").exists()):
    chunks = json.loads((d / "chunks.json").read_text(encoding="utf-8"))["chunks"]
    listing[d] = [c["file"] for c in chunks]
    turns = json.loads((d / "turns.json").read_text(encoding="utf-8"))["turns"]
    short_path, text_path = d / "short_text.json", d / "text.json"
    short = json.loads(short_path.read_text(encoding="utf-8")) if short_path.exists() else {}
    texts = json.loads(text_path.read_text(encoding="utf-8")) if text_path.exists() else {}
    span = {(round(t["start"], 3), round(t["end"], 3)): short[t["file"]] for t in turns if t["file"] in short}
    same = {c["file"]: span[k] for c in chunks if c["file"] not in texts and len(c["parts"]) == 1
            and (k := (round(c["parts"][0][0], 3), round(c["parts"][0][1], 3))) in span}
    if same:
        texts.update(same)
        tmp = d / "text.json.tmp"
        tmp.write_text(json.dumps(dict(sorted(texts.items())), ensure_ascii=False, indent=1), encoding="utf-8")
        os.replace(tmp, text_path)
        reused += len(same)
print(f"{reused} chunks are a short turn transcribed in step 2: that text is reused, not sent again")
failed = transcribe_all(listing, "text.json")

## 5. Dataset

In [ ]:
def is_loop(text):
    """A word or a phrase of up to 4 words more than MAX_REPEATS times in a row ('shu shu shu shu'),
    or one sound repeated into a long run ('O'o'o'o'o'o'o'): Whisper stuck in a loop, or a stutter."""
    if re.search(rf"([^\W\d_]{{1,2}}[{APOSTROPHES}-]?)\1{{5,}}", text, re.IGNORECASE):
        return True
    words = [w.strip(".,!?…:;\"«»“”()[]-").lower() for w in text.split()]
    words = [w for w in words if w]
    for n in range(1, 5):
        for i in range(len(words) - n * (MAX_REPEATS + 1) + 1):
            unit = words[i:i + n]
            if all(words[i + r * n:i + (r + 1) * n] == unit for r in range(1, MAX_REPEATS + 1)):
                return True
    return False


podcasts, items, skipped = [], [], Counter()  # items: (podcast number, source wav, chunk, text)
for d in sorted(p for p in chunks_root.iterdir() if p.is_dir()):
    if not (d / "chunks.json").exists():
        print("skip (no chunks.json, run steps 1-3):", d.name)
        continue
    record = json.loads((d / "chunks.json").read_text(encoding="utf-8"))
    path = d / "text.json"
    done = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    podcasts.append(d.name)
    for c in record["chunks"]:
        text = done.get(c["file"])
        if text is None:
            skipped["not transcribed: run step 4 again"] += 1
            continue
        text = clean(text)
        if not text:
            skipped["empty transcript (no speech heard)"] += 1
        elif c["duration"] < SHORT_SEC and word_count(text) < MIN_WORDS:
            skipped[f"short: under {SHORT_SEC} s and fewer than {MIN_WORDS} words"] += 1
        elif is_loop(text):
            skipped[f"word or phrase more than {MAX_REPEATS} times in a row"] += 1
        else:
            items.append((len(podcasts), d / c["file"], c, text))

missing = [src for _, src, _, _ in items if not src.exists()]
assert not missing, f"{len(missing)} chunks in a chunks.json have no wav, e.g. {missing[:3]}"
print(f"{len(items)} chunks from {len(podcasts)} podcasts, {sum(c['duration'] for _, _, c, _ in items) / 3600:.2f} h")
for why, n in skipped.most_common():
    print(f"  omitted {n} chunks: {why}")
if skipped["not transcribed: run step 4 again"]:
    print("\nWARNING: some chunks have no transcript yet, they are left out of the dataset")

In [ ]:
wav_dir.mkdir(parents=True, exist_ok=True)
for old in [*wav_dir.glob("*.wav"), dataset_dir / "metadata.json", dataset_dir / "metadata.jsonl"]:
    old.unlink(missing_ok=True)  # an earlier dataset

rows = []
for num, src, c, text in tqdm(items, unit="file"):
    dst = wav_dir / f"{num:03d}_{src.name}"  # every podcast has a 0000_speaker_0.wav: podcast number in front
    x, sr = sf.read(src, dtype="float32", always_2d=True)
    if sr != OUT_SR:
        x = AF.resample(torch.from_numpy(x.T.copy()), sr, OUT_SR, **RESAMPLE).T.numpy().clip(-1.0, 1.0)
    sf.write(dst, x, OUT_SR, subtype="PCM_16")
    rows.append({
        "file": f"{wav_dir.name}/{dst.name}",
        "speaker": f"{num:03d}_{c['speaker']}",
        "text": text,
        "duration": round(len(x) / OUT_SR, 3),
        "start": c["start"],
        "end": c["end"],
    })

metadata = {
    "sample_rate": OUT_SR,
    "podcasts": {f"{num:03d}": title for num, title in enumerate(podcasts, 1)},
    "chunks": rows,
}
tmp = dataset_dir / "metadata.json.tmp"
tmp.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
os.replace(tmp, dataset_dir / "metadata.json")  # written last: no metadata.json = the run did not finish
print("saved", len(rows), "chunks to", wav_dir, "and", dataset_dir / "metadata.json")

In [ ]:
# check: metadata and folder match, and a summary of the dataset
meta = json.loads((dataset_dir / "metadata.json").read_text(encoding="utf-8"))
rows = meta["chunks"]
on_disk = {p.name for p in wav_dir.glob("*.wav")}
listed = {Path(m["file"]).name for m in rows}
print("rows:", len(rows), "| wavs:", len(on_disk), "| wav without row:", len(on_disk - listed),
      "| row without wav:", len(listed - on_disk), "| row without text:", sum(not m["text"] for m in rows))
total = sum(m["duration"] for m in rows)
rates = Counter(sf.info(str(wav_dir / name)).samplerate for name in listed & on_disk)
print(f"total {int(total // 3600)}h {int(total % 3600 // 60):02d}m, {len({m['speaker'] for m in rows})} speakers, "
      f"{sum(len(m['text'].split()) for m in rows)} words, sample rates {dict(rates)}")
for lo, hi in zip([0, 1, 2, 5, 10, 20], [1, 2, 5, 10, 20, 30]):
    part = [m["duration"] for m in rows if lo <= m["duration"] < hi]
    print(f"  {lo:>2}-{hi:<2} s: {len(part):5d} chunks, {sum(part) / 3600:5.2f} h")

## Listen to a few chunks
`joined` = an interruption was cut out of it (its wav is shorter than `end - start`).

In [ ]:
import random
from IPython.display import Audio, display

for m in random.sample(meta["chunks"], 5):
    joined = " joined" if m["duration"] < m["end"] - m["start"] - 0.05 else ""
    print(f"{m['file']}  {m['duration']:.1f} s{joined}  {m['text']}")
    display(Audio(str(dataset_dir / m["file"])))